# Comparación de modelos para Paddy Doctor

Este cuaderno compara cuatro clasificadores de las **10 clases** del dataset local:

1. ResNet34 (referencia del benchmark Paddy Doctor).
2. EfficientNet-B0 (baseline del cuaderno original).
3. ViT-Small/16 (rama Transformer).
4. Híbrido EfficientNet-B0 + ViT-Small/16 (fusión de características locales y globales).

La partición estratificada 80/10/10 se crea **antes** de aplicar aumentos y se reutiliza sin cambios para cada modelo. Los aumentos se aplican al vuelo y únicamente al conjunto de entrenamiento; así se evitan copias en disco y fuga de datos. El conjunto de prueba se evalúa una sola vez al final.

## Comparación con los artículos

- El benchmark ACM de Paddy Doctor reporta ResNet34 con F1 de 97,50 % en 16.225 imágenes y 13 clases.
- El artículo de clasificación/detección reporta resultados altos para ViT, pero sus cifras difieren entre resumen, tabla y conclusión; YOLO se evalúa con mAP50 y anotaciones de cajas, una tarea distinta.
- El artículo de Scientific Reports propone MobileNetV3 + GKSO-SA + CatBoost y reporta 98,52 % de accuracy y 97,54 % de F1 con 13.866 imágenes.

Tu descarga de Kaggle tiene 10.407 imágenes y 10 clases. Es una versión/configuración distinta, por lo que las cifras publicadas son referencias, no una comparación directa. Este cuaderno fija un protocolo reproducible para comparar modelos **entre sí** sobre tus mismos ejemplos.

> Recomendado: ejecutar en Google Colab o un entorno con GPU. Los pesos preentrenados se descargan la primera vez. El benchmark queda activado (`RUN_BENCHMARK = True`) y entrena los cuatro modelos al ejecutar las celdas; puede tardar y consumir recursos. C?mbialo a `False` si solo quieres revisar o preparar el cuaderno.

In [ ]:
import gc
import json
import random
import time
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 4
BACKBONE_LR = 1e-4
HEAD_LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Compatible con Windows y clases Dataset definidas en notebooks.
RUN_BENCHMARK = True  # Ejecutar la comparación completa de modelos.

DATA_ROOT = Path("./paddy_data")
CSV_PATH = DATA_ROOT / "train.csv"
TRAIN_DIR = DATA_ROOT / "train_images"
OUTPUT_DIR = Path("./paddy_model_comparison")

if not CSV_PATH.is_file() or not TRAIN_DIR.is_dir():
    raise FileNotFoundError(
        "No encuentro ./paddy_data/train.csv y ./paddy_data/train_images. "
        "Abre este notebook desde la carpeta del proyecto."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
print(f"Dispositivo: {DEVICE} | AMP: {USE_AMP} | torch: {torch.__version__} | timm: {timm.__version__}")
print(f"CSV: {CSV_PATH.resolve()}")

In [ ]:
df = pd.read_csv(CSV_PATH)
required_columns = {"image_id", "label"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas en train.csv: {sorted(missing_columns)}")
if df[list(required_columns)].isna().any().any():
    raise ValueError("Hay valores vacíos en image_id o label.")

df["image_path"] = df.apply(
    lambda row: TRAIN_DIR / str(row["label"]) / str(row["image_id"]), axis=1
)
missing_paths = [str(path) for path in df["image_path"] if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        f"No se encontraron {len(missing_paths)} imágenes. Primeras rutas: {missing_paths[:5]}"
    )

CLASS_NAMES = sorted(df["label"].astype(str).unique())
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
df["label"] = df["label"].astype(str)
df["target"] = df["label"].map(CLASS_TO_INDEX)

train_val_df, test_df = train_test_split(
    df,
    test_size=0.10,
    stratify=df["target"],
    random_state=SEED,
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=1 / 9,
    stratify=train_val_df["target"],
    random_state=SEED,
)

print(f"Total: {len(df):,} | Train: {len(train_df):,} | Validación: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Clases ({len(CLASS_NAMES)}): {CLASS_NAMES}")
display(
    pd.DataFrame(
        {
            "total": df["label"].value_counts().reindex(CLASS_NAMES),
            "train": train_df["label"].value_counts().reindex(CLASS_NAMES),
            "validación": val_df["label"].value_counts().reindex(CLASS_NAMES),
            "test": test_df["label"].value_counts().reindex(CLASS_NAMES),
        }
    ).fillna(0).astype(int)
)

plt.figure(figsize=(11, 5))
sns.countplot(data=df, y="label", order=df["label"].value_counts().index, hue="label", legend=False)
plt.title("Distribución de clases del conjunto completo")
plt.xlabel("Número de imágenes")
plt.ylabel("Clase")
plt.tight_layout()
plt.show()

## Aumentos y conjunto de datos

Los artículos usan aumentos geométricos y de brillo/contraste. Se aplican aquí de forma aleatoria durante cada lectura de una imagen de entrenamiento; cada época ve variaciones distintas. **No se aumentan validación ni prueba**. La transformación es moderada para conservar lesiones pequeñas y sus colores.

In [ ]:
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomAffine(
            degrees=5,
            translate=(0.05, 0.05),
            scale=(0.90, 1.10),
            shear=5,
        ),
        transforms.ColorJitter(brightness=0.10, contrast=0.20, saturation=0.15, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]
)

class PaddyDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["image_path"]) as image:
            image = image.convert("RGB")
            image = self.transform(image)
        return image, int(row["target"])

train_dataset = PaddyDataset(train_df, train_transform)
val_dataset = PaddyDataset(val_df, eval_transform)
test_dataset = PaddyDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

class_counts = train_df["target"].value_counts().reindex(range(len(CLASS_NAMES)), fill_value=0)
if (class_counts == 0).any():
    raise ValueError("Al menos una clase no aparece en la partición de entrenamiento.")
class_weights = len(train_df) / (len(CLASS_NAMES) * class_counts.to_numpy(dtype=np.float32))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print(f"Pesos de clase calculados solo con entrenamiento: {class_weights.cpu().numpy().round(3)}")

sample_images, sample_targets = next(iter(train_loader))
print("Batch aumentado:", tuple(sample_images.shape), "| etiquetas:", tuple(sample_targets.shape))

## Modelos

Los tres modelos individuales usan pesos ImageNet de `timm`. El híbrido concatena las representaciones globales de EfficientNet-B0 y ViT-Small/16 antes de su cabeza clasificadora. Todos reciben las mismas imágenes de 224×224; esta dimensión es nativa para ViT-Small/16 y permite comparar sin cambiar los datos entre modelos.

El entrenamiento es de transferencia de aprendizaje con ajuste fino. La tasa de aprendizaje de los backbones es menor que la de la cabeza para conservar mejor las características preentrenadas. Los pesos de clase compensan la frecuencia desigual sin duplicar imágenes.

In [ ]:
class SingleBackboneClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=num_classes,
        )

    def forward(self, images):
        return self.backbone(images)

class EfficientNetViTHybrid(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )
        self.vit = timm.create_model(
            "vit_small_patch16_224",
            pretrained=True,
            num_classes=0,
        )
        feature_dim = self.cnn.num_features + self.vit.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(p=0.35),
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(p=0.25),
            nn.Linear(512, num_classes),
        )

    def forward(self, images):
        cnn_features = self.cnn(images)
        vit_features = self.vit(images)
        fused_features = torch.cat((cnn_features, vit_features), dim=1)
        return self.classifier(fused_features)

MODEL_SPECS = {
    "ResNet34": "resnet34",
    "EfficientNet-B0": "efficientnet_b0",
    "ViT-Small/16": "vit_small_patch16_224",
    "Híbrido EfficientNet-B0 + ViT-Small/16": "hybrid",
}

def build_model(model_key):
    model_name = MODEL_SPECS[model_key]
    if model_name == "hybrid":
        model = EfficientNetViTHybrid(num_classes=len(CLASS_NAMES))
    else:
        model = SingleBackboneClassifier(model_name, num_classes=len(CLASS_NAMES))
    return model.to(DEVICE)

def make_optimizer(model):
    if isinstance(model, EfficientNetViTHybrid):
        head_parameters = list(model.classifier.parameters())
        backbone_parameters = list(model.cnn.parameters()) + list(model.vit.parameters())
    else:
        head_parameters = list(model.backbone.get_classifier().parameters())
        head_ids = {id(parameter) for parameter in head_parameters}
        backbone_parameters = [
            parameter for parameter in model.parameters() if id(parameter) not in head_ids
        ]
    return torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": BACKBONE_LR},
            {"params": head_parameters, "lr": HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )

## Entrenamiento y selección

La selección del mejor checkpoint y el early stopping usan **F1 macro de validación**. El test no interviene en la selección. Se guarda el mejor estado de cada modelo y el historial de entrenamiento para poder revisar convergencia y reproducir los resultados.

In [ ]:
def autocast_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def collect_predictions(model, loader):
    model.eval()
    targets, predictions = [], []
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=PIN_MEMORY)
            with autocast_context():
                logits = model(images)
            predictions.extend(logits.argmax(dim=1).cpu().numpy().tolist())
            targets.extend(labels.numpy().tolist())
    return np.asarray(targets), np.asarray(predictions)

def score_predictions(targets, predictions):
    return {
        "accuracy": accuracy_score(targets, predictions),
        "balanced_accuracy": balanced_accuracy_score(targets, predictions),
        "f1_macro": f1_score(targets, predictions, average="macro", zero_division=0),
        "f1_weighted": f1_score(targets, predictions, average="weighted", zero_division=0),
    }

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0
    total_examples = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=PIN_MEMORY)
        labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)
    return total_loss / total_examples

def fit_model(model_key):
    model = build_model(model_key)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    scaler = make_grad_scaler()
    safe_name = model_key.lower().replace(" ", "_").replace("/", "_").replace("+", "plus")
    checkpoint_path = OUTPUT_DIR / f"{safe_name}_best.pt"
    history = []
    best_f1 = -1.0
    epochs_without_improvement = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        start_time = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_targets, val_predictions = collect_predictions(model, val_loader)
        val_scores = score_predictions(val_targets, val_predictions)
        scheduler.step(val_scores["f1_macro"])
        epoch_record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_f1_macro": val_scores["f1_macro"],
            "val_accuracy": val_scores["accuracy"],
            "seconds": time.time() - start_time,
        }
        history.append(epoch_record)
        print(
            f"{model_key} | epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"loss={train_loss:.4f} | val_f1_macro={val_scores['f1_macro']:.4f} | "
            f"val_acc={val_scores['accuracy']:.4f} | {epoch_record['seconds']:.1f}s"
        )

        if val_scores["f1_macro"] > best_f1:
            best_f1 = val_scores["f1_macro"]
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping: sin mejora de F1 macro durante {EARLY_STOPPING_PATIENCE} épocas.")
                break

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    test_targets, test_predictions = collect_predictions(model, test_loader)
    test_scores = score_predictions(test_targets, test_predictions)
    report = classification_report(
        test_targets,
        test_predictions,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(
        test_targets,
        test_predictions,
        labels=list(range(len(CLASS_NAMES))),
    )
    del model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "scores": test_scores,
        "report": report,
        "confusion_matrix": cm,
        "history": history,
        "checkpoint": str(checkpoint_path),
        "best_val_f1_macro": best_f1,
    }

## Ejecutar la comparación

La ejecución completa descarga los pesos preentrenados si aún no están en caché, entrena los cuatro modelos y guarda checkpoints y resultados en `paddy_model_comparison/`. En CPU puede tardar mucho; para la comparación completa se recomienda GPU.

Cuando termine, los resultados de test se muestran una vez. No cambies hiperparámetros basándote en test; si ajustas algo, hazlo usando validación y vuelve a reservar un test intacto para la evaluación final.

In [ ]:
def run_benchmark():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_results = {}
    summary_rows = []

    for model_key in MODEL_SPECS:
        print(f"\n{'=' * 80}\nEntrenando: {model_key}\n{'=' * 80}")
        result = fit_model(model_key)
        all_results[model_key] = result
        summary_rows.append(
            {
                "modelo": model_key,
                "test_accuracy": result["scores"]["accuracy"],
                "test_balanced_accuracy": result["scores"]["balanced_accuracy"],
                "test_f1_macro": result["scores"]["f1_macro"],
                "test_f1_weighted": result["scores"]["f1_weighted"],
                "best_val_f1_macro": result["best_val_f1_macro"],
            }
        )
        with open(OUTPUT_DIR / f"{model_key.lower().replace(' ', '_')}_report.json", "w", encoding="utf-8") as file:
            json.dump(result["report"], file, ensure_ascii=False, indent=2)

    results_df = pd.DataFrame(summary_rows).sort_values("test_f1_macro", ascending=False)
    results_df.to_csv(OUTPUT_DIR / "comparison_results.csv", index=False)
    with open(OUTPUT_DIR / "run_config.json", "w", encoding="utf-8") as file:
        json.dump(
            {
                "seed": SEED,
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "epochs_max": NUM_EPOCHS,
                "class_names": CLASS_NAMES,
                "split_sizes": {
                    "train": len(train_df),
                    "validation": len(val_df),
                    "test": len(test_df),
                },
            },
            file,
            ensure_ascii=False,
            indent=2,
        )
    return all_results, results_df

if RUN_BENCHMARK:
    RESULTS, RESULTS_TABLE = run_benchmark()
    display(RESULTS_TABLE.style.format({column: "{:.4f}" for column in RESULTS_TABLE.columns if column != "modelo"}))
else:
    print("Entrenamiento no iniciado. Cambia RUN_BENCHMARK = True en la primera celda de configuración.")

In [ ]:
if RUN_BENCHMARK:
    fig, axes = plt.subplots(2, 2, figsize=(16, 13))
    for axis, (model_key, result) in zip(axes.flat, RESULTS.items()):
        sns.heatmap(
            result["confusion_matrix"],
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            ax=axis,
        )
        axis.set_title(f"Test: {model_key}")
        axis.set_xlabel("Predicción")
        axis.set_ylabel("Etiqueta real")
        axis.tick_params(axis="x", rotation=75)
        axis.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for axis, (model_key, result) in zip(axes.flat, RESULTS.items()):
        history = pd.DataFrame(result["history"])
        axis.plot(history["epoch"], history["train_loss"], label="Pérdida train")
        axis.plot(history["epoch"], 1 - history["val_f1_macro"], label="1 - F1 macro val")
        axis.set_title(model_key)
        axis.set_xlabel("Época")
        axis.legend()
    plt.tight_layout()
    plt.show()

    for model_key, result in RESULTS.items():
        print(f"\nReporte de clasificación (test): {model_key}")
        display(
            pd.DataFrame(result["report"])
            .T.reindex(CLASS_NAMES + ["macro avg", "weighted avg"])
            .round(4)
        )

## Lectura de resultados y mejoras

- Prioriza **F1 macro** y recall por clase, además de accuracy: así las clases menos frecuentes no quedan ocultas por las más comunes.
- Si el híbrido no mejora, no es un fracaso: compara el costo de inferencia, tamaño del modelo y resultados por clase; en un dataset relativamente pequeño, un modelo individual puede generalizar mejor.
- Grad-CAM puede añadirse después para explicar qué regiones influyen en las predicciones. No equivale a anotación ni a detección de lesiones.
- Para una comparación académica estricta, documenta versión exacta del dataset, semilla, particiones, resolución, pesos preentrenados y hardware. Los artículos usan cantidades/clases distintas y algunos presentan discrepancias en sus métricas.